# Using damek-edge for edge computing with EP5G

Minimal working example to connect `damek-edge` to user-equipment in R1 over EP5G.

As a demonstration, a container is deployed on a worker, and connected to 5G using an Advantech router. This is optional, and can be skipped if other user-equipment is to be used instead.

A router is used to connect the `ep5g-net` and `telenor-net` networks, but one additional step is required for user equipment to be reachable from `damek-edge`. The necessary commands are provided in this notebook.

The remote host `damek-edge` has also been configured to provide public internet access to user-equipment through network address translation.

Authentication

In [ ]:
import os, re

with open('nils-project-openrc.sh', 'r') as f:
    script_content = f.read()
    pattern = r'export\s+(\w+)\s*=\s*("[^"]+"|[^"\n]+)'
    matches = re.findall(pattern, script_content)

    for name, value in matches:
        os.environ[name] = value.strip('"')

# password read from file

Install required packages and dependencies. Ignore the warnings.

In [ ]:
!pip uninstall -q -y moviepy
!pip install -q jedi
!pip install -q git+https://github.com/KTH-EXPECA/python-chi

Import packages and define custom functions

In [ ]:
import json
from loguru import logger
import chi.network, chi.container
from chi.expeca import reserve, list_reservations, unreserve_byid, get_container_status, wait_until_container_removed, get_available_publicips, get_worker_interfaces, get_segment_ids

def get_reservation_id_by_name(name):
    for lease in list_reservations(brief=True):
        if name in lease['name']:
            return lease['reservation_id']

def get_available_interface(worker_name, number=1):
    interfaces = list(get_worker_interfaces(worker_name).values())[0]
    available_ifs = []
    for interface in interfaces.keys():
        if len(interfaces[interface]['connections']) == 0:
            available_ifs.append(interface)
    if len(available_ifs) < number:
        logger.info(f"{json.dumps(interfaces, indent=4)}")
        raise Exception(f"Did not find enough interfaces on {worker_name}")
    return sorted(available_ifs)[:number]

def get_network_id_by_name(name):
    for network in chi.network.list_networks():
        if name in network['name']:
            return network['id']
    raise Exception(f"Network {name} not found")

def get_segment_id(name):
    result = get_segment_ids(name)
    return next(iter(result.values()))

Project settings

In [ ]:
# Fixed IP addresses
ep5g_gw_addr = "10.30.111.10"
public_gw_addr = "130.237.11.97"
telenor_snet_gw_addr = "10.141.0.1"
telenor_damek_edge_addr = "10.2.58.104"
telenor_net_cidr = "10.2.58.0/24"
ue_gw_addr = "10.42.3.1"
ue_priv_addr = "10.42.3.2"
ue_cidr = "172.16.0.0/16"
adv_nat_addr = {
    "adv-01": "172.16.0.8",
    "adv-02": "172.16.0.88",
    "adv-03": "172.16.0.40",
    "adv-04": "172.16.0.96",
    "adv-05": "172.16.0.64",
    "adv-06": "172.16.0.72",
    "adv-07": "172.16.0.104",
    "adv-08": "172.16.0.56",
}
turtlebot_addr = "172.16.0.192"

# Routes
route_from_ue_to_telenor = "10.2.58.0/24-10.42.3.1"     # 10.2.58.0/24 via 10.42.3.1

Choose equipment

In [ ]:
ue1_node_worker = "worker-03"
ue1_adv_name = "adv-05"

Reserve the required equipment and resources

In [ ]:
experiment_duration = {"days": 0, "hours": 2}

# List of required leases with details
required_leases = [
    { 
        "type": "network",
        "name": "ep5g",
        "net_name": "ep5g-vip",
        "segment_id": get_segment_id("ep5g"),
        "duration": experiment_duration
    },
    {
        "type": "network",
        "name": ue1_adv_name,
        "net_name": ue1_adv_name,
        "segment_id": get_segment_id(ue1_adv_name),
        "duration": experiment_duration,
    },
    {
        "type": "device",
        "name": ue1_node_worker,
        "duration": experiment_duration,
    },
]

# List of previously existing leases
existing_leases = list_reservations(brief=True)

# Reserve outstanding resources
for required_lease in required_leases:
    lease_name_with_suffix = required_lease["name"] + "-lease"

    # Check if the resource is already leased
    is_already_leased = False
    for existing_lease in existing_leases:
        if existing_lease["name"] == lease_name_with_suffix:
            is_already_leased = True
            break

    # If it is already leased, check the lease status
    if is_already_leased and existing_lease["status"] == "ACTIVE":
        logger.info(f"Resource {required_lease['name']} is already leased and ACTIVE.")
        continue
    if is_already_leased and existing_lease["status"] == "TERMINATED":
        logger.info(f"Removing TERMINATED lease of {existing_lease['name']} before proceeding.")
        unreserve_byid(existing_lease["id"])
    
    # If it is NOT already leased, reserve it (default case)
    reserve(required_lease)

Create networks and routers

In [ ]:
# Create router between epg5 and telenor-net
ep5g_net = chi.network.get_network("ep5g-vip-net")

try:
    chi.network.get_router("edge-router")
    logger.info("Router already exists.")
except:
    router = chi.network.create_router("edge-router", "telenor-net")
    ext_gw_addr = router['external_gateway_info']['external_fixed_ips'][0]['ip_address']
    chi.network.add_subnet_to_router(router["id"], ep5g_net["subnets"][0])
    chi.network.add_route_to_router(router["id"], ue_cidr, ep5g_gw_addr)
    chi.network.add_route_to_router(router["id"], "0.0.0.0/0", telenor_damek_edge_addr)
    logger.success(f"Router created with external gateway address {ext_gw_addr}.")

Get control strings for configuring damek-edge forwarding

In [ ]:
logger.warning(f"If running the ue-node below, make sure to run the following on damek-edge first:")
logger.info(f"\"sudo ip route del {adv_nat_addr[ue1_adv_name]}; sudo ip route add {adv_nat_addr[ue1_adv_name]} via {ext_gw_addr}\"")
logger.warning(f"If using the turtlebot, run the following on damek-edge:")
logger.info(f"\"sudo ip route del {turtlebot_addr}; sudo ip route add {turtlebot_addr} via {ext_gw_addr}\"")

Optional: Start UE1

In [ ]:
ue_node_interfaces = get_available_interface(ue1_node_worker)
logger.info(f"Using interface(s): {ue_node_interfaces}")

ue_node_container_name = "ue-node"
ue_node_image_name = "nilsjor/ros-humble-turtlebot:edge-server-husarnet-v2.4"

with open("husarnet-joincode.txt", 'r') as file:
    ue_node_env_vars = {
        "HUSARNET_JOIN_CODE": file.read().strip(),
        "HUSARNET_HOSTNAME": ue_node_container_name,
        "HOSTNAME": ue_node_container_name,
        "DNS_IP": "1.1.1.1",
        "GATEWAY_IP": ue_gw_addr,
    }

ue_node_labels = {
    "networks.1.interface": ue_node_interfaces[0],
    "networks.1.ip": ue_priv_addr + "/24",
    "networks.1.routes": route_from_ue_to_telenor,
    "networks.1.gateway": ue_gw_addr,
    "capabilities.privileged": "true",
}

try:
    chi.container.destroy_container(ue_node_container_name)
    wait_until_container_removed(ue_node_container_name)
    logger.success("Previous container destroyed.")
except:
    logger.info("No previous container found.")

ue_node_container = chi.container.create_container(
    name = ue_node_container_name,
    image = ue_node_image_name,
    reservation_id = get_reservation_id_by_name(ue1_node_worker),
    environment = ue_node_env_vars,
    mounts = [
        { 'source': 'husarnet-config-device', 'destination': '/var/lib/husarnet' },
    ],
    nets = [
        { "network": get_network_id_by_name(ue1_adv_name) },
    ],
    labels = ue_node_labels,
)

chi.container.wait_for_active(ue_node_container_name)
logger.success("Container deployed and active.")

Optional: Remove default route to disconnect UE1 from the public internet

In [ ]:
chi.container.execute(ue_node_container_name, "ip route del default")
print(chi.container.execute(ue_node_container_name, "ip route show")["output"])

## Teardown

Destroy all containers

In [ ]:
try:
    status = get_container_status(ue_node_container_name)
    chi.container.destroy_container(ue_node_container_name)
    wait_until_container_removed(ue_node_container_name)
except:
    logger.info("No ue-node container found.")

logger.info("Stopped and removed all containers")

Proceed to clean up the rest of the project.

In [ ]:
# find the router again
router = None
try:
    router = chi.network.get_router("edge-router")
except Exception as ex:
    logger.info("Could not find edge-router.")

if router:
    # remove all routes from the router
    chi.network.remove_all_routes_from_router(router["id"])
    logger.success("Removed all routes from router.")

    # remove all subnets from the router
    subnets = chi.network.list_subnets()
    logger.info(f"Checking all {len(subnets)} subnets.")
    for subnet in subnets:
        try:
            chi.network.remove_subnet_from_router(router["id"],subnet["id"])
        except Exception as ex:
            pass
    logger.success("Removed all subnets from router")

    chi.network.delete_router(router["id"])
    logger.success("Deleted the router")

Terminate reservations

In [ ]:
leaseslist = list_reservations(brief=True)
for lease in leaseslist:
    unreserve_byid(lease["id"])
    logger.success("Removed " + lease["name"])

logger.info("no leases remaining.")